# Spynach quick start

[Spynach](https://ajerschow.github.io/spynach/) is a Python reimplementation of the
[Spinach](https://spindynamics.org/) spin dynamics kernel, verified stage by stage
against the MATLAB original. This notebook installs it and runs the example from the
project page: a two-spin system at 14.1 T, pulse-acquire on the protons.

Run the cells in order (**Runtime → Run all**). The install takes about a minute on Colab.

In [ ]:
%pip install -q "spynach[plot] @ git+https://github.com/ajerschow/spynach.git"

## A two-spin system

`create` builds the spin system from the isotopes and interactions (shifts in ppm,
J couplings in Hz), `basis` chooses the formalism, and `state` makes the initial
state and the detection state. `liquid` runs the `acquire` pulse sequence in the
liquid-state NMR context and returns the free induction decay.

In [ ]:
import numpy as np

import spynach

spin_system = spynach.create(
    {"isotopes": ["1H", "1H"], "magnet": 14.1},
    {"zeeman": {"scalar": [1.00, 1.20]}, "coupling": {"scalar": [[0, 12.0], [0, 0]]}},
)
spin_system = spynach.basis(spin_system, {"formalism": "zeeman-hilb", "approximation": "none"})
pars = {
    "spins": ["1H"],
    "offset": 660,
    "sweep": 500,
    "npoints": 1024,
    "rho0": spynach.state(spin_system, "L+", "1H"),
    "coil": spynach.state(spin_system, "L+", "1H"),
}
fid = spynach.liquid(spin_system, spynach.acquire, pars, "nmr")
spectrum = np.fft.fftshift(np.fft.fft(fid, 4096))
print(f"FID: {fid.shape[0]} complex points; spectrum: {spectrum.shape[0]} points")

## The spectrum

`plot_1d` draws the spectrum on a ppm axis built from `offset`, `sweep` and the
zero-filled length, decreasing from left to right as is usual in NMR. The figure is
interactive: hover to read off shifts and intensities, drag to zoom in, double-click
to zoom out. The two protons at 1.00 and 1.20 ppm are 120 Hz apart at 14.1 T, ten
times their 12 Hz coupling, so the spectrum is a pair of doublets.

In [ ]:
from spynach.plotting import plot_1d

plot_1d(spin_system, spectrum.real, pars, title="Two coupled protons at 14.1 T")

## Where next

- The [Hilbert space guide](https://github.com/ajerschow/spynach/blob/main/docs/hilbert-space.md)
  continues from this example: the simulation pipeline, pulse sequences, tolerances and
  parity with Spinach.
- [Coming from Spinach](https://github.com/ajerschow/spynach/blob/main/docs/coming-from-spinach.md)
  lists what carries over from Spinach and what differs.
- The [project page](https://ajerschow.github.io/spynach/) links every guide, its
  notebook and the Spinach comparison reports.

To run any other guide notebook here, open it from
[docs/notebooks](https://github.com/ajerschow/spynach/tree/main/docs/notebooks) in Colab
and add the install cell above at the top.